# Camada Silver — Limpeza, ETL e Feature Engineering
**Responsável:** F3 · **Aprovador:** F1  
Projeto Final de Big Data — DataRoute Recife (UNINASSAU, 6º noturno)

> Regra: use apenas os caminhos de `src/config.py`. Nada de `C:/...` ou `/content/drive/...`.

In [1]:
# Setup padrão (NÃO ALTERAR): funciona local e no Google Colab
import sys
from pathlib import Path
try:
    import google.colab  # noqa: F401
    if not Path('dataroute-recife').exists() and not Path('src').exists():
        !git clone -q https://github.com/pcsdv0/dataroute-recife.git
    if Path('dataroute-recife').exists():
        %cd dataroute-recife
    !pip install -q -r requirements.txt
except ImportError:
    pass
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.config import *          # caminhos relativos oficiais
from src.utils import ler_csv_robusto, salvar_csv
if not SILVER_SINISTROS.exists():  # gera Bronze -> Silver -> Gold se ainda não existir
    from src.pipeline import executar
    executar()

In [2]:
from src import bronze, silver
from src.contratos import validar_silver
res = silver.run(bronze.listar_arquivos())
validar_silver(res['df'])
res['relatorio']

{'linhas_entrada': 72255,
 'colunas_entrada': 45,
 'nulos_entrada': 1733304,
 'bairros_distintos_entrada': 126,
 'situacao_cancelada_removidas': 11563,
 'situacao_duplicidade_removidas': 455,
 'duplicatas_exatas_removidas': 4,
 'sem_data_valida_removidas': 0,
 'datas_ano_iso_corrigidas': 32,
 'datas_fora_do_ano_do_arquivo_removidas': 0,
 'sem_hora_mantidas': 2442,
 'nulos_numericos_imputados_zero': 402053,
 'classificacao_inferida': 28,
 'classificacao_nao_sinistro_removidas': 8,
 'vitimasfatais_corrigidas_pela_classificacao': 24,
 'vitimas_ajustadas_para_1': 212,
 'linhas_saida': 60225,
 'colunas_saida': 27,
 'nulos_saida': 4884,
 'bairros_distintos_saida': 93}

In [3]:
from src.pipeline import tabela_antes_depois
tabela_antes_depois({'silver': res['relatorio']})

,Indicador,Antes (Bronze),Depois (Silver)
0,Registros (chamados -> sinistros),72255,60225
1,Colunas (união dos layouts anuais -> contrato),45,27
2,Valores nulos,1733304,4884
3,Chamados CANCELADA,11563,0
4,Chamados DUPLICIDADE,455,0
5,Duplicatas exatas,4,0
6,"Chamados que não são sinistro (APOIO, OUTROS...)",8,0
7,Registros sem data válida,0,0
8,Datas com ano errado (bug ano-semana ISO) corr...,32,0
9,Datas fora do ano do arquivo,0,0


In [4]:
# Bairros após a padronização (deve dar 93: 92 dos 94 bairros oficiais + NAO INFORMADO;
# Pau-Ferro e Peixinhos não têm nenhuma ocorrência registrada na base da CTTU)
print(res['df']['bairro'].nunique())
res['df']['bairro'].value_counts()

93


bairro
BOA VIAGEM             7392
IMBIRIBEIRA            3727
SANTO AMARO            3053
AFOGADOS               2300
BOA VISTA              2272
                       ... 
POCO DA PANELA           30
CACOTE                   30
SITIO DOS PINTOS         29
CORREGO DO JENIPAPO      27
BREJO DE BEBERIBE        24
Name: count, Length: 93, dtype: int64

In [5]:
# Refinamento F3 (Wagner): nomes de agentes digitados no campo 'bairro'
# ("MARCELO" em 2015 e "FABIO" em 2017) passam a NAO INFORMADO.
# A regra já está incorporada em src/silver.py (CORRECAO_BAIRROS); aqui só conferimos.
assert silver.CORRECAO_BAIRROS["MARCELO"] == "NAO INFORMADO"
assert silver.CORRECAO_BAIRROS["FABIO"] == "NAO INFORMADO"
assert not res["df"]["bairro"].isin(["MARCELO", "FABIO"]).any()

print(f"Bairros distintos após refinamento F3: {res['df']['bairro'].nunique()}")
print("\nDistribuição da Escala de Severidade (UPS):")
print(res["df"]["indice_severidade"].value_counts().sort_index())

# Tabela 'Antes vs. Depois' oficial (93 bairros distintos)
display(tabela_antes_depois({"silver": res["relatorio"]}))

Bairros distintos após refinamento F3: 93

Distribuição da Escala de Severidade (UPS):
indice_severidade
1     38646
4     19567
6      1736
13      276
Name: count, dtype: int64


,Indicador,Antes (Bronze),Depois (Silver)
0,Registros (chamados -> sinistros),72255,60225
1,Colunas (união dos layouts anuais -> contrato),45,27
2,Valores nulos,1733304,4884
3,Chamados CANCELADA,11563,0
4,Chamados DUPLICIDADE,455,0
5,Duplicatas exatas,4,0
6,"Chamados que não são sinistro (APOIO, OUTROS...)",8,0
7,Registros sem data válida,0,0
8,Datas com ano errado (bug ano-semana ISO) corr...,32,0
9,Datas fora do ano do arquivo,0,0
